<a href="https://colab.research.google.com/github/shamikkarkhanis/AV-SSL-Optimization-JEPA/blob/main/hw_5_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1

In [ ]:
import copy
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
# setting constants
HF_DATASET_ID = "ilee0022/GTSRB"
IMAGE_SIZE = 32
INCEPTION_IMAGE_SIZE = 299
BATCH_SIZE = 64
INCEPTION_BATCH_SIZE = 16
NUM_EPOCHS = 15
INCEPTION_NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.1
NUM_WORKERS = 2

CACHE_DIR = Path(".cache")
HF_CACHE_DIR = CACHE_DIR / "huggingface"
TORCH_CACHE_DIR = CACHE_DIR / "torch"

for cache_path in [CACHE_DIR, HF_CACHE_DIR, TORCH_CACHE_DIR]:
    cache_path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE_DIR / "datasets")
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BASELINE_CNN_PATH = CHECKPOINT_DIR / "baseline_traffic_sign_cnn.pt"
INCEPTION_PATH = CHECKPOINT_DIR / "inception_v3_gtsrb.pt"
AUGMENTED_CNN_PATH = CHECKPOINT_DIR / "augmented_traffic_sign_cnn.pt"


In [ ]:
dataset = load_dataset(HF_DATASET_ID)
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/77.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39209 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12630 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'label'],
        num_rows: 39209
    })
    test: Dataset({
        features: ['image', 'Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'label'],
        num_rows: 12630
    })
})

In [ ]:
dataset["train"][0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=29x30>,
 'Width': 29,
 'Height': 30,
 'Roi.X1': 5,
 'Roi.Y1': 6,
 'Roi.X2': 24,
 'Roi.Y2': 25,
 'label': 0}

In [ ]:
train_valid = dataset["train"].train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds_hf = train_valid["train"]
val_ds_hf = train_valid["test"]
test_ds_hf = dataset["test"]

sample_keys = list(train_ds_hf.features.keys())

if "image" in sample_keys:
    image_key = "image"
else:
    raise KeyError(f"Could not find an image column. Available columns: {sample_keys}")

if "label" in sample_keys:
    label_key = "label"
else:
    raise KeyError(f"Could not find a label column. Available columns: {sample_keys}")

num_classes = len(set(train_ds_hf[label_key]))
print(f"Train samples: {len(train_ds_hf)}")
print(f"Validation samples: {len(val_ds_hf)}")
print(f"Test samples: {len(test_ds_hf)}")
print(f"Image column: {image_key}")
print(f"Label column: {label_key}")
print(f"Number of classes: {num_classes}")

Train samples: 35288
Validation samples: 3921
Test samples: 12630
Image column: image
Label column: label
Number of classes: 43


In [ ]:
def build_transform(image_size, mean, std, augment=False):
    transform_steps = [transforms.Resize((image_size, image_size))]

    if augment:
        transform_steps.extend([
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
        ])

    transform_steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])
    return transforms.Compose(transform_steps)


CNN_MEAN = [0.5, 0.5, 0.5]
CNN_STD = [0.5, 0.5, 0.5]
INCEPTION_MEAN = [0.485, 0.456, 0.406]
INCEPTION_STD = [0.229, 0.224, 0.225]

# transforming to model requirements w/ and wo/ augmentation
cnn_eval_transform = build_transform(IMAGE_SIZE, CNN_MEAN, CNN_STD, augment=False)
cnn_augmented_transform = build_transform(IMAGE_SIZE, CNN_MEAN, CNN_STD, augment=True)
inception_eval_transform = build_transform(INCEPTION_IMAGE_SIZE, INCEPTION_MEAN, INCEPTION_STD, augment=False)

In [ ]:
class HFGTSRBDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, image_key="image", label_key="label"):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.image_key = image_key
        self.label_key = label_key

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]

        image = sample[self.image_key]
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        else:
            image = image.convert("RGB")

        label = int(sample[self.label_key])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available()
}


def build_dataloaders(train_transform, eval_transform, batch_size):
    train_dataset = HFGTSRBDataset(train_ds_hf, transform=train_transform, image_key=image_key, label_key=label_key)
    val_dataset = HFGTSRBDataset(val_ds_hf, transform=eval_transform, image_key=image_key, label_key=label_key)
    test_dataset = HFGTSRBDataset(test_ds_hf, transform=eval_transform, image_key=image_key, label_key=label_key)

    current_loader_kwargs = dict(loader_kwargs)
    current_loader_kwargs["batch_size"] = batch_size

    train_loader = DataLoader(train_dataset, shuffle=True, **current_loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **current_loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **current_loader_kwargs)
    return train_loader, val_loader, test_loader


baseline_train_loader, baseline_val_loader, baseline_test_loader = build_dataloaders(
    cnn_eval_transform,
    cnn_eval_transform,
    BATCH_SIZE
)

inception_train_loader, inception_val_loader, inception_test_loader = build_dataloaders(
    inception_eval_transform,
    inception_eval_transform,
    INCEPTION_BATCH_SIZE
)

augmented_train_loader, augmented_val_loader, augmented_test_loader = build_dataloaders(
    cnn_augmented_transform,
    cnn_eval_transform,
    BATCH_SIZE
)

len(baseline_train_loader), len(inception_train_loader), len(augmented_train_loader)

(552, 2206, 552)

In [ ]:
# checking model shapes
baseline_images, baseline_labels = next(iter(baseline_train_loader))
inception_images, inception_labels = next(iter(inception_train_loader))
print("Baseline batch:", baseline_images.shape, baseline_labels.shape)
print("Inception batch:", inception_images.shape, inception_labels.shape)

Baseline batch: torch.Size([64, 3, 32, 32]) torch.Size([64])
Inception batch: torch.Size([16, 3, 299, 299]) torch.Size([16])


In [ ]:
inception_weights = models.Inception_V3_Weights.DEFAULT

class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def build_custom_cnn(num_classes):
    return TrafficSignCNN(num_classes=num_classes).to(device)


def build_inception_model(num_classes):
    model = models.inception_v3(weights=inception_weights, aux_logits=True)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    for parameter in model.AuxLogits.fc.parameters():
        parameter.requires_grad = True
    for parameter in model.fc.parameters():
        parameter.requires_grad = True
    return model.to(device)


build_custom_cnn(num_classes)

TrafficSignCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()


## Model Structure Reasoning
- three block setup
- block 1: 3 -> 32 channels which learn low level features, boundaries, edges, curves
- block 2: 32 -> 64 channels which learn more meaningful sign features such as shapes and number regions
- block 3: 64 -> 128 channels learn higher level features suh as sign structure
- maxpool reduces spacial size after each block which makes representations more robust and lowers computation as it shrinks the spacial size of the feature maps
- the two conv layers before each pooling layer gives the model more opportunities to capture spacial information than one conv layer would allow
- using dropout in the classifier head allows the model to reduce overfitting as traffic signs can get repetitive

In [ ]:
criterion = nn.CrossEntropyLoss()


def build_optimizer_and_scheduler(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
    return optimizer, scheduler

In [ ]:
def get_logits_and_loss(model, images, labels, criterion, is_training):
    outputs = model(images)

    if isinstance(outputs, tuple):
        main_logits = outputs[0]
        aux_logits = outputs[1]
    elif hasattr(outputs, "logits"):
        main_logits = outputs.logits
        aux_logits = getattr(outputs, "aux_logits", None)
    else:
        main_logits = outputs
        aux_logits = None

    loss = criterion(main_logits, labels)
    if is_training and aux_logits is not None:
        loss = loss + 0.4 * criterion(aux_logits, labels)

    return main_logits, loss


def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_training):
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits, loss = get_logits_and_loss(model, images, labels, criterion, is_training)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, checkpoint_path, experiment_name):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_weights = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"[{experiment_name}] Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, checkpoint_path)

    model.load_state_dict(best_weights)
    return model, history


def run_experiment(experiment_name, model_builder, loaders, checkpoint_path, num_epochs):
    train_loader, val_loader, test_loader = loaders
    model = model_builder(num_classes)
    optimizer, scheduler = build_optimizer_and_scheduler(model)

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        checkpoint_path=checkpoint_path,
        experiment_name=experiment_name
    )

    return {
        "name": experiment_name,
        "model": model,
        "history": history,
        "checkpoint_path": checkpoint_path,
        "test_loader": test_loader
    }


In [ ]:
baseline_results = run_experiment(
    experiment_name="Baseline custom CNN",
    model_builder=build_custom_cnn,
    loaders=(baseline_train_loader, baseline_val_loader, baseline_test_loader),
    checkpoint_path=BASELINE_CNN_PATH,
    num_epochs=NUM_EPOCHS
)

inception_results = run_experiment(
    experiment_name="InceptionV3",
    model_builder=build_inception_model,
    loaders=(inception_train_loader, inception_val_loader, inception_test_loader),
    checkpoint_path=INCEPTION_PATH,
    num_epochs=INCEPTION_NUM_EPOCHS
)

augmented_results = run_experiment(
    experiment_name="Augmented custom CNN",
    model_builder=build_custom_cnn,
    loaders=(augmented_train_loader, augmented_val_loader, augmented_test_loader),
    checkpoint_path=AUGMENTED_CNN_PATH,
    num_epochs=NUM_EPOCHS
)

experiment_results = {
    "baseline_cnn": baseline_results,
    "inception_v3": inception_results,
    "augmented_cnn": augmented_results
}

for result in experiment_results.values():
    print(f"Saved best checkpoint for {result['name']}: {result['checkpoint_path']}")

[Baseline custom CNN] Epoch 1/15 | train_loss=2.6898 train_acc=0.2307 | val_loss=1.2888 val_acc=0.5539
[Baseline custom CNN] Epoch 2/15 | train_loss=0.8584 train_acc=0.7088 | val_loss=0.2714 val_acc=0.9133
[Baseline custom CNN] Epoch 3/15 | train_loss=0.3117 train_acc=0.8966 | val_loss=0.1166 val_acc=0.9653
[Baseline custom CNN] Epoch 4/15 | train_loss=0.1680 train_acc=0.9462 | val_loss=0.0711 val_acc=0.9801
[Baseline custom CNN] Epoch 5/15 | train_loss=0.1018 train_acc=0.9676 | val_loss=0.0593 val_acc=0.9855
[Baseline custom CNN] Epoch 6/15 | train_loss=0.0780 train_acc=0.9761 | val_loss=0.0368 val_acc=0.9908
[Baseline custom CNN] Epoch 7/15 | train_loss=0.0693 train_acc=0.9794 | val_loss=0.0325 val_acc=0.9918
[Baseline custom CNN] Epoch 8/15 | train_loss=0.0556 train_acc=0.9829 | val_loss=0.0295 val_acc=0.9918
[Baseline custom CNN] Epoch 9/15 | train_loss=0.0517 train_acc=0.9842 | val_loss=0.0504 val_acc=0.9883
[Baseline custom CNN] Epoch 10/15 | train_loss=0.0425 train_acc=0.9872 | 

In [ ]:
def predict_dataset(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            logits, _ = get_logits_and_loss(model, images, labels.to(device), criterion, is_training=False)
            preds = logits.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)


def evaluate_experiment(result):
    test_labels, test_preds = predict_dataset(result["model"], result["test_loader"])
    test_accuracy = accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds)
    cm = confusion_matrix(test_labels, test_preds)

    result["test_labels"] = test_labels
    result["test_preds"] = test_preds
    result["test_accuracy"] = test_accuracy
    result["classification_report"] = report
    result["confusion_matrix"] = cm
    return result

In [ ]:
for experiment_key, result in experiment_results.items():
    result = evaluate_experiment(result)
    experiment_results[experiment_key] = result

    print(f"{result['name']} test accuracy: {result['test_accuracy']:.4f}")
    print("Classification report:")
    print(result["classification_report"])


Baseline custom CNN test accuracy: 0.9689
Classification report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98        60
           1       0.96      1.00      0.98       720
           2       0.98      1.00      0.99       750
           3       0.98      0.97      0.98       450
           4       0.99      0.99      0.99       660
           5       0.92      0.96      0.94       630
           6       0.98      0.75      0.85       150
           7       0.98      0.98      0.98       450
           8       0.97      0.97      0.97       450
           9       0.98      1.00      0.99       480
          10       1.00      1.00      1.00       660
          11       0.94      0.99      0.96       420
          12       0.99      0.95      0.97       690
          13       0.98      1.00      0.99       720
          14       1.00      1.00      1.00       270
          15       0.96      1.00      0.98       210
          16    

In [ ]:
summary = {
    key: {
        "name": result["name"],
        "test_accuracy": round(result["test_accuracy"], 4),
        "checkpoint": str(result["checkpoint_path"])
    }
    for key, result in experiment_results.items()
}
summary

{'baseline_cnn': {'name': 'Baseline custom CNN',
  'test_accuracy': 0.9689,
  'checkpoint': 'checkpoints/baseline_traffic_sign_cnn.pt'},
 'inception_v3': {'name': 'InceptionV3',
  'test_accuracy': 0.6678,
  'checkpoint': 'checkpoints/inception_v3_gtsrb.pt'},
 'augmented_cnn': {'name': 'Augmented custom CNN',
  'test_accuracy': 0.9683,
  'checkpoint': 'checkpoints/augmented_traffic_sign_cnn.pt'}}

## Custom vs Pretrained Model Differences
#### CNN:
- faster to optimize
- stable convergence accross training
- better results on every class

#### Inception:
- lower recall on many classes
- underfit the data

This is probably due to a lack of true fine-tuning using the Inception model. We merely loaded and froze all the weights, then only trained the final classifier layer. If we replaced the classifier with our own classification layer and unfroze some of the final Inception layers and trained those with our traffic sign data we would expect to see much better results. This just shows the imageNet dataset which the Inception model was trained on is very different than a specialized datset like traffic signs.

## Dataset Augmentation Differences
- There was no meaningful difference when introducting data augmentation in terms of testing accuracy
- We did see improvement in recall for certain classes but also saw a decrease in some areas
- without augmentation our baseline model was able to converge faster and with a higher accuracy than with augmentation